In [1]:
# !pip -q install -U segmentation-models-pytorch==0.3.3 timm "albumentations>=1.4.0,<1.5.0" opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 40.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 25.4 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [2]:
import os, random, glob, time
import numpy as np
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [3]:
# ==========================================
# 1. INITIALIZATION & SEEDING
# ==========================================
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [11]:
# ==========================================
# 2. DATASET PARSING & RIGOROUS SPLIT
# ==========================================
KVASIR_IMAGES_DIR = "/kaggle/input/datasets/debeshjha1/kvasirseg/Kvasir-SEG/Kvasir-SEG/images"
KVASIR_MASKS_DIR  = "/kaggle/input/datasets/debeshjha1/kvasirseg/Kvasir-SEG/Kvasir-SEG/masks"
CVC_ROOT = "/kaggle/input/datasets/balraj98/cvcclinicdb/PNG"
CVC_IMAGES_DIR = os.path.join(CVC_ROOT, "Original")
CVC_MASKS_DIR  = os.path.join(CVC_ROOT, "Ground Truth")

def list_image_mask_pairs(images_dir, masks_dir, img_exts=(".jpg",".jpeg",".png")):
    img_files = []
    for e in img_exts:
        img_files.extend(glob.glob(os.path.join(images_dir, f"*{e}")))
    img_files = sorted(img_files)
    mask_files = sorted(glob.glob(os.path.join(masks_dir, "*")))
    mask_map = {os.path.splitext(os.path.basename(m))[0]: m for m in mask_files}
    
    pairs = []
    for im in img_files:
        stem = os.path.splitext(os.path.basename(im))[0]
        if stem in mask_map:
            pairs.append((im, mask_map[stem]))
        else:
            for k in mask_map.keys():
                if k.startswith(stem) or stem.startswith(k):
                    pairs.append((im, mask_map[k]))
                    break
    return pairs

kvasir_pairs = list_image_mask_pairs(KVASIR_IMAGES_DIR, KVASIR_MASKS_DIR)
cvc_pairs = list_image_mask_pairs(CVC_IMAGES_DIR, CVC_MASKS_DIR)

print(f"Kvasir pairs mapped: {len(kvasir_pairs)}")
print(f"CVC pairs mapped: {len(cvc_pairs)}")

def split_protocol_A(pairs, seed=SEED):
    idx = np.arange(len(pairs))
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    test_idx = idx[:100]
    trainval_idx = idx[100:1000]
    
    trainval = [pairs[i] for i in trainval_idx]
    test = [pairs[i] for i in test_idx]
    
    rng2 = np.random.default_rng(seed + 1)
    tv_idx = np.arange(len(trainval))
    rng2.shuffle(tv_idx)
    
    val_idx = tv_idx[:100]
    train_idx = tv_idx[100:]
    return {"train": [trainval[i] for i in train_idx], "val": [trainval[i] for i in val_idx], "test": test}

splits = split_protocol_A(kvasir_pairs)

Kvasir pairs mapped: 1000
CVC pairs mapped: 612


In [16]:
# ==========================================
# 3. TRANSFORMS & DATALOADER
# ==========================================
train_tfms = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5), 
    A.Affine(scale=(0.9, 1.1), translate_percent=(-0.05, 0.05), rotate=(-15, 15), p=0.5), 
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(),
    ToTensorV2()
])

val_tfms = A.Compose([
    A.Resize(512, 512),
    A.Normalize(),
    ToTensorV2()
])

class PolypSegDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        mask = (cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)

        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"]

        return img, mask.unsqueeze(0)

In [6]:
# ==========================================
# 4. ARCHITECTURE: SEGFORMER + SCSE
# ==========================================
model = smp.Unet(
    encoder_name="mit_b3", 
    encoder_weights="imagenet", 
    in_channels=3, 
    classes=1, 
    activation=None,
    decoder_attention_type="scse"
).to(device)
print(f"Architecture: SCSE-UNet | Encoder: mit_b3 | Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

Downloading: "https://github.com/qubvel/segmentation_models.pytorch/releases/download/v0.0.2/mit_b3.pth" to /root/.cache/torch/hub/checkpoints/mit_b3.pth


100%|██████████| 170M/170M [00:01<00:00, 127MB/s]  


Architecture: SCSE-UNet | Encoder: mit_b3 | Params: 47.48M


In [20]:
# ==========================================
# 5. THE WAVELET FREQUENCY MODULE & LOSS
# ==========================================
class HaarDWT(nn.Module):
    def __init__(self):
        super().__init__()
        weight = torch.tensor([
            [[[0.5,  0.5], [ 0.5,  0.5]]],   
            [[[-0.5, -0.5], [ 0.5,  0.5]]],  
            [[[-0.5,  0.5], [-0.5,  0.5]]],  
            [[[0.5, -0.5], [-0.5,  0.5]]]    
        ])
        self.register_buffer('weight', weight)
        
    def forward(self, x):
        return F.conv2d(x, self.weight, stride=2, padding=0)

dwt_module = HaarDWT().to(device)
lovasz_loss = smp.losses.LovaszLoss(mode='binary')
bce_loss = nn.BCEWithLogitsLoss(reduction='none')

# Notice w_wav defaults to 0.0 now
def wavelet_lovasz_loss(logits, targets, w_lovasz=0.5, w_bce=0.4, w_wav=0.0):
    l_lovasz = lovasz_loss(logits, targets)
    bce_full = bce_loss(logits, targets).mean()
    
    # Only calculate Wavelet loss if the weight is > 0 (Saves GPU memory in early epochs)
    if w_wav > 0:
        pred_probs = torch.sigmoid(logits)
        pred_dwt = dwt_module(pred_probs)
        target_dwt = dwt_module(targets)
        # L1 Loss strictly on High-Frequency Edge Maps (Indices 1, 2, 3)
        l_wavelet = F.l1_loss(pred_dwt[:, 1:], target_dwt[:, 1:])
        return w_lovasz * l_lovasz + w_bce * bce_full + w_wav * l_wavelet
    else:
        return w_lovasz * l_lovasz + w_bce * bce_full

In [21]:
# ==========================================
# 6. ENGINE & TRAINING LOOP
# ==========================================
@torch.no_grad()
def evaluate_metrics(logits, targets, thr=0.5, eps=1e-7):
    preds = (torch.sigmoid(logits) > thr).float().view(logits.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    inter = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1)
    return ((2.0 * inter + eps) / (union + eps)).mean().item(), ((inter + eps) / (union - inter + eps)).mean().item()

def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    return (np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H),
            np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H))

EPOCHS = 40
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda')

# Added current_w_wav parameter to the training step
def run_epoch(loader, train=True, cutmix_prob=0.0, img_size=384, current_w_wav=0.0):
    model.train() if train else model.eval()
    epoch_loss, epoch_dice, epoch_iou, n = 0, 0, 0, 0

    for imgs, masks in loader:
        imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        
        if imgs.shape[-1] != img_size:
            imgs = F.interpolate(imgs, size=(img_size, img_size), mode='bilinear', align_corners=False)
            masks = F.interpolate(masks, size=(img_size, img_size), mode='nearest')
        
        if train and np.random.rand() < cutmix_prob:
            lam = np.random.beta(1.0, 1.0)
            rand_index = torch.randperm(imgs.size(0)).to(device)
            bbx1, bby1, bbx2, bby2 = rand_bbox(imgs.size(), lam)
            
            imgs[:, :, bbx1:bbx2, bby1:bby2] = imgs[rand_index, :, bbx1:bbx2, bby1:bby2]
            masks_a, masks_b = masks, masks[rand_index]
            
            with torch.set_grad_enabled(True):
                with torch.amp.autocast('cuda'):
                    logits = model(imgs)
                    loss = wavelet_lovasz_loss(logits, masks_a, w_wav=current_w_wav) * lam + wavelet_lovasz_loss(logits, masks_b, w_wav=current_w_wav) * (1. - lam)
        else:
            with torch.set_grad_enabled(train):
                with torch.amp.autocast('cuda'):
                    logits = model(imgs)
                    loss = wavelet_lovasz_loss(logits, masks, w_wav=current_w_wav)
            
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        d, j = evaluate_metrics(logits.detach(), masks.detach())
        bs = imgs.size(0)
        epoch_loss += loss.item() * bs
        epoch_dice += d * bs
        epoch_iou += j * bs
        n += bs

    return epoch_loss / n, epoch_dice / n, epoch_iou / n

train_loader = DataLoader(PolypSegDataset(splits["train"], train_tfms), batch_size=8, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(PolypSegDataset(splits["val"], val_tfms), batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
kvasir_test_loader = DataLoader(PolypSegDataset(splits["test"], val_tfms), batch_size=1, shuffle=False, num_workers=2)
cvc_test_loader = DataLoader(PolypSegDataset(cvc_pairs, val_tfms), batch_size=1, shuffle=False, num_workers=2)

best_dice = -1
print("Initiating Curriculum-Guided Wavelet Pipeline...")

for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    
    # --- CURRICULUM CONTROLLER ---
    if epoch <= 20:
        # Phase 1: Establish global shape (No Wavelets)
        curr_size, curr_cutmix, curr_wav = 256, 0.5, 0.0
    elif epoch <= 35:
        # Phase 2: Refine boundaries (Activate Wavelets)
        curr_size, curr_cutmix, curr_wav = 384, 0.0, 0.3
    else:
        # Phase 3: High-Res Edge Tracing (Heavy Wavelets)
        curr_size, curr_cutmix, curr_wav = 512, 0.0, 0.5
        if epoch == 36:
            print(">>> FREEZING ENCODER: Transitioning to High-Res Wavelet Fine-Tuning <<<")
            for param in model.encoder.parameters(): param.requires_grad = False

    tl, td, tj = run_epoch(train_loader, train=True, cutmix_prob=curr_cutmix, img_size=curr_size, current_w_wav=curr_wav)
    vl, vd, vj = run_epoch(val_loader, train=False, cutmix_prob=0.0, img_size=384, current_w_wav=0.0) # Validation loss remains standard
    scheduler.step()
    
    improved = vd > best_dice
    if improved:
        best_dice = vd
        torch.save(model.state_dict(), "best_model.pt")
        
    dt = time.time() - t0
    print(f"Ep {epoch:02d} | Res: {curr_size} | Wav: {curr_wav:.1f} | T_Dice={td:.4f} | V_Dice={vd:.4f} | {'BEST' if improved else ''} | {dt:.1f}s")

Initiating Curriculum-Guided Wavelet Pipeline...
Ep 01 | Res: 256 | Wav: 0.0 | T_Dice=0.6723 | V_Dice=0.5813 | BEST | 26.1s
Ep 02 | Res: 256 | Wav: 0.0 | T_Dice=0.6730 | V_Dice=0.8179 | BEST | 25.8s
Ep 03 | Res: 256 | Wav: 0.0 | T_Dice=0.6821 | V_Dice=0.7700 |  | 24.6s
Ep 04 | Res: 256 | Wav: 0.0 | T_Dice=0.7221 | V_Dice=0.8039 |  | 24.9s
Ep 05 | Res: 256 | Wav: 0.0 | T_Dice=0.7207 | V_Dice=0.8239 | BEST | 25.5s
Ep 06 | Res: 256 | Wav: 0.0 | T_Dice=0.7107 | V_Dice=0.8272 | BEST | 25.3s
Ep 07 | Res: 256 | Wav: 0.0 | T_Dice=0.6827 | V_Dice=0.7987 |  | 25.2s
Ep 08 | Res: 256 | Wav: 0.0 | T_Dice=0.7171 | V_Dice=0.8316 | BEST | 25.5s
Ep 09 | Res: 256 | Wav: 0.0 | T_Dice=0.7006 | V_Dice=0.7264 |  | 24.9s
Ep 10 | Res: 256 | Wav: 0.0 | T_Dice=0.7061 | V_Dice=0.8407 | BEST | 25.4s
Ep 11 | Res: 256 | Wav: 0.0 | T_Dice=0.7136 | V_Dice=0.8553 | BEST | 25.5s
Ep 12 | Res: 256 | Wav: 0.0 | T_Dice=0.7245 | V_Dice=0.7815 |  | 25.6s
Ep 13 | Res: 256 | Wav: 0.0 | T_Dice=0.7288 | V_Dice=0.8542 |  | 25.0s


In [22]:
# ==========================================
# 7. FINAL EVALUATION
# ==========================================
print("\n========== FINAL TEST BENCHMARKS ==========")
model.load_state_dict(torch.load("best_model.pt", weights_only=True))

def evaluate_dataset(loader, dataset_name):
    model.eval()
    dices, ious = [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            # Evaluate test sets at benchmark 384 size
            imgs = F.interpolate(imgs, size=(384, 384), mode='bilinear', align_corners=False)
            masks = F.interpolate(masks, size=(384, 384), mode='nearest')
            with torch.amp.autocast('cuda'): logits = model(imgs)
            d, j = evaluate_metrics(logits, masks)
            dices.append(d); ious.append(j)
    print(f"{dataset_name} -> Dice: {np.mean(dices):.4f} | IoU: {np.mean(ious):.4f}")

evaluate_dataset(kvasir_test_loader, "Kvasir-SEG (Internal)")
evaluate_dataset(cvc_test_loader, "CVC-ClinicDB (External)")


========== FINAL TEST BENCHMARKS ==========
Kvasir-SEG (Internal) -> Dice: 0.9060 | IoU: 0.8458
CVC-ClinicDB (External) -> Dice: 0.8453 | IoU: 0.7724
